In [ ]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------------------------
# 0) 경로·파일 이름 설정
# --------------------------------------------------------------------
DATA_DIR = Path(r"D:/fintech/data")                  # ↩ 직접 경로 입력
SRC_FILE = DATA_DIR / "PSIDSHELF_1968_2021_LONG.dta" # 원본 Stata
DST_PARQ = DATA_DIR / "psidshelf_long_full.parquet"  # 전체 데이터 저장 (권장)
DST_CSV  = DATA_DIR / "psidshelf_long_full.csv"      # 필요 시 CSV
DST_COLS = DATA_DIR / "psidshelf_columns.txt"        # 컬럼 목록

# --------------------------------------------------------------------
# 1) WIDE 파일 전체 불러오기
#    • convert_categoricals=False → Stata 범주형을 숫자로 유지
# --------------------------------------------------------------------
df = pd.read_stata(SRC_FILE, convert_categoricals=False)

# --------------------------------------------------------------------
# 2) 그대로 저장
# --------------------------------------------------------------------
df.to_parquet(DST_PARQ, index=False)   # Parquet (빠른 I/O·압축률↑)
# df.to_csv(DST_CSV, index=False)      # ← 주석 해제 시 CSV도 동시에 저장

# --------------------------------------------------------------------
# 3) 컬럼 목록만 따로 저장해두기
# --------------------------------------------------------------------
pd.Series(df.columns).to_csv(DST_COLS, index=False, header=False)

print(f"✅ 전체 데이터 저장 완료 → {DST_PARQ}")
print(f"   행/열 크기      : {df.shape[0]:,} × {df.shape[1]}")
print(f"✅ 컬럼 목록 저장   → {DST_COLS}")


In [2]:
import pandas as pd
from pathlib import Path

# 0) 파일 경로 설정
SRC_FILE = Path(r"D:/fintech/data/PSIDSHELF_1968_2021_LONG.dta")
DST_CSV  = Path(r"D:/fintech/data/psidshelf_long_full.csv")

# 1) 청크 크기(행 개수) 지정
chunksize = 200_000

# 2) Stata 파일 읽기용 iterator 생성
reader = pd.read_stata(
    str(SRC_FILE),
    convert_categoricals=False,  # 범주형 변수도 숫자로 유지
    iterator=True,
    chunksize=chunksize
)

# 3) 첫 번째 청크부터 순차적으로 CSV에 쓰기
first_chunk = True
for df_chunk in reader:
    # (필요 시 타입 변환)
    # df_chunk = df_chunk.astype("float64")
    
    # CSV로 저장: 첫 청크만 header, 쓰기 모드는 첫 번째만 'w', 나머지는 'a'
    df_chunk.to_csv(
        str(DST_CSV),
        index=False,
        header=first_chunk,
        mode="w" if first_chunk else "a",
        encoding="utf-8-sig"  # 한글 깨짐 방지
    )
    first_chunk = False

print("✅ .dta → .csv 변환 완료:", DST_CSV)


✅ .dta → .csv 변환 완료: D:\fintech\data\psidshelf_long_full.csv


In [15]:
import dask.dataframe as dd

# 1) Parquet 읽기 (작업 중에도 메모리에 안 올림)
ddf = dd.read_parquet("D:/fintech/data/psidshelf_wide.parquet", engine="pyarrow")

# 2) 예: WIDE→LONG 변환
id_vars    = ["ID", "YEAR", "FUID", "PNUM"]  # 예시: 고유 식별자 컬럼들
value_vars = [c for c in ddf.columns if c.startswith("EARN_TOT")]
long = dd.melt(ddf,
               id_vars=id_vars,
               value_vars=value_vars,
               var_name="earn_year",
               value_name="earn_value")

# 3) 결과도 Parquet으로 저장 (메모리 걱정 無)
long.to_parquet("D:/fintech/data/psidshelf_long_dd.parquet",
                write_index=False)


ImportError: An error occurred while calling the read_parquet method registered to the pandas backend.
Original Message: The pyarrow installation is not built with support for 'dataset' (DLL load failed while importing _dataset: 지정된 프로시저를 찾을 수 없습니다.)

In [12]:
# split_psid_dta.py
"""
PSIDSHELF_1968_2021_WIDE.dta  →  Parquet(또는 Feather) 다중 파일로
메모리 친화적인 스트리밍 분할 저장 스크립트
"""
from pathlib import Path
import pyreadstat                    # Stata → pandas (row_offset / row_limit 지원)
import pyarrow.parquet as pq
import pyarrow as pa

def split_save(src_path: str,
               dst_dir : str,
               chunk_rows: int = 25_000,
               fmt: str = "parquet",
               cast_float32: bool = True):
    """
    Parameters
    ----------
    src_path : str  · 원본 .dta 파일
    dst_dir  : str  · 출력 디렉터리 (자동 생성)
    chunk_rows : int  · 한 번에 처리할 행 수
    fmt : "parquet" | "feather"
    cast_float32 : True → float64 → float32 로 메모리 절감
    """
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    # ── 1. 총 행 수만 메타데이터로 확인 ──────────────────
    _, meta = pyreadstat.read_dta(src_path, metadata_only=True)
    total_rows = meta.number_rows
    print(f"📦  Total rows        : {total_rows:,}")
    print(f"📂  Saving to         : {dst_dir.resolve()}")
    print(f"🚚  Chunk size (rows) : {chunk_rows:,}\n")

    part = 1
    for start in range(0, total_rows, chunk_rows):
        # ── 2. 부분(chunk) 로딩 ───────────────────────────
        df, _ = pyreadstat.read_dta(src_path,
                                    row_offset=start,
                                    row_limit=chunk_rows)

        if cast_float32:
            float64_cols = df.select_dtypes("float64").columns
            df[float64_cols] = df[float64_cols].astype("float32")

        # ── 3. 저장 ─────────────────────────────────────
        fname = dst_dir / f"psid_part_{part:03d}.{ 'parquet' if fmt=='parquet' else 'feather'}"
        if fmt == "parquet":
            table = pa.Table.from_pandas(df, preserve_index=False)
            pq.write_table(table, fname, compression="zstd")
        else:  # feather
            df.reset_index(drop=True).to_feather(fname)

        print(f"✅  Part {part:03d}  ·  rows {len(df):>6,}  →  {fname.name}")
        part += 1

    print("\n🎉  All done – files ready for 업로드!")

# ── 4. 직접 실행 예시 ─────────────────────────────────────
if __name__ == "__main__":
    split_save(
        src_path=r"D:/fintech/data/PSIDSHELF_1968_2021_WIDE.dta",
        dst_dir =r"D:/fintech/data/psid_parts",
        chunk_rows=25_000,      # RAM 상황에 따라 10k~50k 로 조정
        fmt="parquet"           # 'feather' 로도 가능
    )


TypeError: read_dta() got an unexpected keyword argument 'metadata_only'

In [13]:
import re, numpy as np, pandas as pd
from pathlib import Path

DATA_DIR = Path(r"D:/fintech/data")
df = pd.read_stata(DATA_DIR / "PSIDSHELF_1968_2021_WIDE.dta",
                   convert_categoricals=False)

# -------------------------------------------------------------------
# 1) 변수별로 사용할 wave 목록 정의 ----------------------------------
#    - 길이를 바꾸면 '평균 n개' 윈도우 폭이 바뀜
# -------------------------------------------------------------------
WAVES_5  = [2021, 2019, 2017, 2015, 2013]        # 최신 5개
WAVES_3  = [2021, 2019, 2017]

ROOTS_LEVEL   = ["EARN_TOT_RD", "FINC_TOT_RDF",
                 "WLTH_TOT_NET_RDF",            # 자산
                 "DEP_SCORE_TOT", "ADL_SUM_TOT"]
ROOTS_TREND3  = ["EARN_TOT_RD", "DEP_SCORE_TOT"]
ROOTS_TREND5  = ["CCON_HIBP_DIAG_ANY", "CCON_ARTH_DIAG_ANY"]

# -------------------------------------------------------------------
# 2) 헬퍼: root와 연도 리스트로 실제 컬럼명 반환 ----------------------
# -------------------------------------------------------------------
def cols(root, years):
    return [f"{root}_{y}" for y in years if f"{root}_{y}" in df.columns]

# -------------------------------------------------------------------
# 3) 레벨: 최근 비결측 1개 백필 --------------------------------------
# -------------------------------------------------------------------
for r in ROOTS_LEVEL:
    cand = cols(r, WAVES_5)
    if not cand: continue
    df[f"{r}_lvl"] = df[cand].bfill(axis=1).iloc[:, 0]   # 왼쪽→오른쪽 백필

# -------------------------------------------------------------------
# 4) 3-wave 기울기 & 5-wave 평균/표준편차 -----------------------------
# -------------------------------------------------------------------
for r in ROOTS_TREND3:
    c = cols(r, WAVES_3)
    if len(c) >= 2:
        # (마지막-첫)/abs(첫)  → 최근 4년간 상대 기울기
        df[f"{r}_slope3"] = (df[c[0]] - df[c[-1]]) / df[c[-1]].abs()

for r in ROOTS_TREND5:
    c = cols(r, WAVES_5)
    if len(c) >= 2:
        df[f"{r}_mean5"] = df[c].mean(1)
        df[f"{r}_std5"]  = df[c].std(1)

# -------------------------------------------------------------------
# 5) 최종 feature 집합 -----------------------------------------------
# -------------------------------------------------------------------
pattern = re.compile(r"_(lvl|slope3|mean5|std5)$")
features = [c for c in df.columns if pattern.search(c)]
df_feat  = df[["ID"] + features]

print("완성된 feature 수:", len(features))
print(df_feat.sample(3).T.head(20))


[경고] DEBT_TOT_RDF_xxxx 형태 컬럼을 찾을 수 없습니다.
최종 선택 컬럼 수: 16
   DEMO_SEX  DEMO_BIRTH_YEAR  EARN_TOT_RD_2021  FINC_TOT_RDF_2021  \
0       1.0           1915.0               NaN                NaN   
1       2.0           1922.0               NaN                NaN   
2       1.0           1947.0               NaN                NaN   
3       2.0           1948.0               NaN                NaN   
4       2.0           1972.0               NaN                NaN   

   WLTH_TOT_NET_RDF_2021  EMP_WORK_2021  EDU_LEVEL_MAX  HOME_STAT_2021  \
0                    NaN            NaN            0.0             NaN   
1                    NaN            NaN            0.0             NaN   
2                    NaN            NaN            1.0             NaN   
3                    NaN            NaN            1.0             NaN   
4                    NaN            NaN            NaN             NaN   

   HOME_OWN_VAL_RDF_2021  CCON_ARTH_DIAG_ANY_2021  CCON_HIBP_DIAG_ANY_2021  \
0     

# Home Credit 데이터셋 모델 생성

#### train 데이터셋만 이용

In [22]:
"""
Home Credit Default Risk – LightGBM 모델(A)  
(※ application_train.csv 만으로 5-Fold CV AUC 측정)

────────────────────────────────────────
구조
1) 데이터 로드
2) 범주형 → One-Hot Encoding①
3) LightGBM 5-Fold CV (Early-Stopping 포함)
   · 파라미터/기술 선택 이유는 코드 하단 ‘각주’ 참조
────────────────────────────────────────
"""

import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from lightgbm import LGBMClassifier
from packaging import version

# 1) ── 데이터 로드 ───────────────────────────────
DATA_PATH = r'D:/fintech/data/home-credit-default-risk/application_train.csv'
df = pd.read_csv(DATA_PATH)

y = df.pop('TARGET')                       # 레이블
cat_cols = df.select_dtypes('object').columns  # 범주형 컬럼

# 2) ── One-Hot Encoding (train 데이터만) ──────────
ohe = OneHotEncoder(handle_unknown='ignore',  # unseen 범주 ↦ 올-0 벡터②
                    sparse_output=False)
ohe.fit(df[cat_cols])

X_num = df.drop(columns=cat_cols).astype(np.float32).values
X_cat = ohe.transform(df[cat_cols]).astype(np.float32)
X     = np.hstack([X_num, X_cat])

feature_names = (list(df.drop(columns=cat_cols).columns) +
                 list(ohe.get_feature_names_out(cat_cols)))

# 3) ── LightGBM 5-Fold CV ───────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(y), dtype=np.float32)

# ▶ 중요도 누적용 배열  (추가)
import_sum = np.zeros(len(feature_names), dtype=np.float32)

use_callbacks = version.parse(lgb.__version__) >= version.parse("4.0.0")

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    clf = LGBMClassifier(
        n_estimators=2000,         # 충분한 트리 수 + 조기중단③
        learning_rate=0.05,        # 작은 step → 과적합 완화④
        num_leaves=31,             # depth≈5 수준⑤
        colsample_bytree=0.80,     # 변수 샘플링으로 다양성↑⑥
        subsample=0.80,            # Row bagging → 과적합↓⑦
        reg_lambda=0.1,            # L2 정규화⑧
        random_state=42 + fold     # Fold마다 다른 seed⑨
    )

    fit_params = dict(
        eval_set=[(X[val_idx], y.iloc[val_idx])],
        eval_metric='auc'
    )
    if use_callbacks:                      # LightGBM ≥ 4.0
        fit_params['callbacks'] = [
            lgb.early_stopping(200, verbose=False),  # 200회 개선 없으면 중단⑩
            lgb.log_evaluation(period=200)
        ]
    else:                                  # LightGBM ≤ 3.x
        fit_params.update(early_stopping_rounds=200, verbose=200)

    clf.fit(X[tr_idx], y.iloc[tr_idx], **fit_params)

    best_iter = clf.best_iteration_
    oof[val_idx] = clf.predict_proba(X[val_idx],
                                     num_iteration=best_iter)[:, 1]
    # ▶ 중요도 누적  (추가)
    import_sum += clf.feature_importances_ / kf.n_splits

    gc.collect()

print(f"\n▶ 5-Fold CV AUC = {roc_auc_score(y, oof):.5f}")

# ▶ TOP-20 Feature Importance 출력  (추가)
imp_df = (pd.DataFrame({'feature': feature_names,
                        'importance': import_sum})
            .sort_values('importance', ascending=False)
            .head(20))
print("\n[top-20 features]")
print(imp_df.to_string(index=False))

# ───────────────────────────────────────────────
# 각주: 파라미터·기술 선택 이유
# ───────────────────────────────────────────────
# ① 원-핫 인코딩만으로도 범주형 처리가 간단하며,
#    추가 Target Encoding 없이도 0.75+ AUC 달성 가능.
# ② handle_unknown='ignore' → 검증 Fold에 train에 없던
#    범주가 나와도 에러 대신 올-0 벡터로 처리.
# ③ n_estimators=2000 + Early-Stopping: 넉넉히 잡아두고
#    최적 트리 개수를 자동 탐색 → 과적합 방지·편의성↑
# ④ learning_rate 0.05: 작은 학습률로 안정적 학습.
# ⑤ num_leaves=31 (LightGBM 기본): 깊이≈5, 과적합 위험 낮음.
# ⑥ colsample_bytree 0.8: 각 트리마다 20% 변수 무작위 제외
#    → 트리 다양성 확보·일반화 성능 개선.
# ⑦ subsample 0.8: Row 샘플링으로 모델 분산↓, 과적합↓
# ⑧ reg_lambda 0.1: L2 패널티로 leaf weight 폭주 제어.
# ⑨ Fold마다 seed 변경 → 앙상블 효과로 예측 분산↓
# ⑩ 200 round 동안 AUC 개선 없으면 학습 중단.


[LightGBM] [Info] Number of positive: 19876, number of negative: 226132
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.107763 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11698
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 241
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080794 -> initscore=-2.431606
[LightGBM] [Info] Start training from score -2.431606
[200]	valid_0's auc: 0.75997	valid_0's binary_logloss: 0.245228
[400]	valid_0's auc: 0.76053	valid_0's binary_logloss: 0.244928


C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 19888, number of negative: 226121
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.325925 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11690
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 241
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080843 -> initscore=-2.430954
[LightGBM] [Info] Start training from score -2.430954
[200]	valid_0's auc: 0.759396	valid_0's binary_logloss: 0.245252
[400]	valid_0's auc: 0.760413	valid_0's binary_logloss: 0.244934


C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 19743, number of negative: 226266
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.391641 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11672
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 241
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080253 -> initscore=-2.438912
[LightGBM] [Info] Start training from score -2.438912
[200]	valid_0's auc: 0.755957	valid_0's binary_logloss: 0.250545
[400]	valid_0's auc: 0.756621	valid_0's binary_logloss: 0.250393
[600]	valid_0's auc: 0.757004	valid_0's binary_logloss: 0.250295


...

[LightGBM] [Info] Number of positive: 19921, number of negative: 226088
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.321281 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11701
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 240
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080977 -> initscore=-2.429150
[LightGBM] [Info] Start training from score -2.429150
[200]	valid_0's auc: 0.756728	valid_0's binary_logloss: 0.244261
[400]	valid_0's auc: 0.757169	valid_0's binary_logloss: 0.244019
[600]	valid_0's auc: 0.756671	valid_0's binary_logloss: 0.244118


C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 19872, number of negative: 226137
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.331796 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11705
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 239
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080778 -> initscore=-2.431829
[LightGBM] [Info] Start training from score -2.431829
[200]	valid_0's auc: 0.758246	valid_0's binary_logloss: 0.24596
[400]	valid_0's auc: 0.758139	valid_0's binary_logloss: 0.245944


C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



▶ 5-Fold CV AUC = 0.75879

[top-20 features]
                   feature  importance
              EXT_SOURCE_3  697.200012
              EXT_SOURCE_1  643.599976
              EXT_SOURCE_2  546.200012
                DAYS_BIRTH  515.000000
                AMT_CREDIT  457.400024
               AMT_ANNUITY  452.600037
           DAYS_ID_PUBLISH  392.600006
           AMT_GOODS_PRICE  378.800018
             DAYS_EMPLOYED  373.600006
    DAYS_LAST_PHONE_CHANGE  342.000000
         DAYS_REGISTRATION  339.800018
                SK_ID_CURR  302.800018
          AMT_INCOME_TOTAL  256.200012
               OWN_CAR_AGE  238.200012
REGION_POPULATION_RELATIVE  215.399994
AMT_REQ_CREDIT_BUREAU_YEAR  161.000000
   HOUR_APPR_PROCESS_START  145.800003
            TOTALAREA_MODE  121.199997
              LANDAREA_AVG  116.199997
          BASEMENTAREA_AVG   93.000000


In [23]:
from pathlib import Path
import joblib      # 또는 pickle

SAVE_DIR = Path(r"D:/fintech/models")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ① One-Hot Encoder 저장
joblib.dump(ohe, SAVE_DIR / "ohe_homecredit.pkl")

# ② 각 Fold별 LightGBM 저장 ──
for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    ...
    clf.fit(X[tr_idx], y.iloc[tr_idx], **fit_params)

    # booster_ 객체를 그대로 저장하면 best_iteration 도 함께 보존됨
    clf.booster_.save_model(
        str(SAVE_DIR / f"lgbm_fold{fold:02d}.txt")
    )
    # 필요하면 joblib 로도 저장 가능
    joblib.dump(clf, SAVE_DIR / f"lgbm_fold{fold:02d}.pkl")


[LightGBM] [Info] Number of positive: 19876, number of negative: 226132
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.112462 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11698
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 242
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080794 -> initscore=-2.431606
[LightGBM] [Info] Start training from score -2.431606
[200]	valid_0's auc: 0.80181	valid_0's binary_logloss: 0.233368
[400]	valid_0's auc: 0.834392	valid_0's binary_logloss: 0.222709
[600]	valid_0's auc: 0.859144	valid_0's binary_logloss: 0.213389
[800]	valid_0's auc: 0.879725	valid_0's binary_logloss: 0.20492
[1000]	valid_0's auc: 0.896547	valid_0's binary_logloss: 0.19713
[1200]	valid_0's auc: 0.909779	valid_0's binary_logloss: 0.190139
[1400]	valid_0's auc: 0.922018	valid_0's binary_l

In [20]:
import re, numpy as np, pandas as pd
from pathlib import Path

DATA_DIR = Path(r"D:/fintech/data")
df = pd.read_stata(DATA_DIR / "PSIDSHELF_1968_2021_WIDE.dta",
                   convert_categoricals=False)

# -------------------------------------------------------------------
# 1) 변수별로 사용할 wave 목록 정의 ----------------------------------
#    - 길이를 바꾸면 '평균 n개' 윈도우 폭이 바뀜
# -------------------------------------------------------------------
WAVES_5  = [2021, 2019, 2017, 2015, 2013]        # 최신 5개
WAVES_3  = [2021, 2019, 2017]

ROOTS_LEVEL   = ["EARN_TOT_RD", "FINC_TOT_RDF",
                 "WLTH_TOT_NET_RDF",            # 자산
                 "DEP_SCORE_TOT", "ADL_SUM_TOT"]
ROOTS_TREND3  = ["EARN_TOT_RD", "DEP_SCORE_TOT"]
ROOTS_TREND5  = ["CCON_HIBP_DIAG_ANY", "CCON_ARTH_DIAG_ANY"]

# -------------------------------------------------------------------
# 2) 헬퍼: root와 연도 리스트로 실제 컬럼명 반환 ----------------------
# -------------------------------------------------------------------
def cols(root, years):
    return [f"{root}_{y}" for y in years if f"{root}_{y}" in df.columns]

# -------------------------------------------------------------------
# 3) 레벨: 최근 비결측 1개 백필 --------------------------------------
# -------------------------------------------------------------------
for r in ROOTS_LEVEL:
    cand = cols(r, WAVES_5)
    if not cand: continue
    df[f"{r}_lvl"] = df[cand].bfill(axis=1).iloc[:, 0]   # 왼쪽→오른쪽 백필

# -------------------------------------------------------------------
# 4) 3-wave 기울기 & 5-wave 평균/표준편차 -----------------------------
# -------------------------------------------------------------------
for r in ROOTS_TREND3:
    c = cols(r, WAVES_3)
    if len(c) >= 2:
        # (마지막-첫)/abs(첫)  → 최근 4년간 상대 기울기
        df[f"{r}_slope3"] = (df[c[0]] - df[c[-1]]) / df[c[-1]].abs()

for r in ROOTS_TREND5:
    c = cols(r, WAVES_5)
    if len(c) >= 2:
        df[f"{r}_mean5"] = df[c].mean(1)
        df[f"{r}_std5"]  = df[c].std(1)

# -------------------------------------------------------------------
# 5) 최종 feature 집합 -----------------------------------------------
# -------------------------------------------------------------------
pattern = re.compile(r"_(lvl|slope3|mean5|std5)$")
features = [c for c in df.columns if pattern.search(c)]
df_feat  = df[["ID"] + features]

print("완성된 feature 수:", len(features))
print(df_feat.sample(3).T.head(20))


C:\Users\Admin\AppData\Local\Temp\ipykernel_15192\4148857571.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{r}_lvl"] = df[cand].bfill(axis=1).iloc[:, 0]   # 왼쪽→오른쪽 백필
C:\Users\Admin\AppData\Local\Temp\ipykernel_15192\4148857571.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{r}_lvl"] = df[cand].bfill(axis=1).iloc[:, 0]   # 왼쪽→오른쪽 백필
C:\Users\Admin\AppData\Local\Temp\ipykernel_15192\4148857571.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` ma

완성된 feature 수: 11
                           48         71827          10698
ID                        4172.0  6781004.0  935185.000000
EARN_TOT_RD_lvl              NaN        NaN   63728.474110
FINC_TOT_RDF_lvl             NaN        NaN   69056.233297
WLTH_TOT_NET_RDF_lvl         NaN        NaN  114499.827343
DEP_SCORE_TOT_lvl            NaN        NaN            NaN
ADL_SUM_TOT_lvl              NaN        NaN       0.000000
EARN_TOT_RD_slope3           NaN        NaN            NaN
DEP_SCORE_TOT_slope3         NaN        NaN            NaN
CCON_HIBP_DIAG_ANY_mean5     NaN        NaN       0.000000
CCON_HIBP_DIAG_ANY_std5      NaN        NaN       0.000000
CCON_ARTH_DIAG_ANY_mean5     NaN        NaN       0.000000
CCON_ARTH_DIAG_ANY_std5      NaN        NaN       0.000000


C:\Users\Admin\AppData\Local\Temp\ipykernel_15192\4148857571.py:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{r}_mean5"] = df[c].mean(1)
C:\Users\Admin\AppData\Local\Temp\ipykernel_15192\4148857571.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{r}_std5"]  = df[c].std(1)


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
import joblib, gc

# 0) 경로
DATA_DIR = Path(r"D:/fintech/data")
psid_path = DATA_DIR / "PSIDSHELF_1968_2021_WIDE.dta"
hc_model_path = Path(r"D:/fintech/models/ohe_homecredit.pkl")  # 이미 저장해 둔 모델 (exist_ok)

# 1) PSID 데이터 로드
df = pd.read_stata(psid_path, convert_categoricals=False)

# 2) 매핑 딕셔너리 (HomeCredit ↔ PSID)
mapping = {
    "EXT_SOURCE_1":  "WLTH_TOT_DEB_ND",
    "EXT_SOURCE_2":  "WLTH_TOT_NET_ND",
    "EXT_SOURCE_3":  "FINC_TOT_ND",
    "DAYS_BIRTH":    "DEMO_AGE_GEN",
    "CODE_GENDER":   "DEMO_SEX",
    "NAME_EDUCATION_TYPE": "EDU_LEVEL",
    "EMP_STAT":      "EMP_STAT_1M",       # 또는 EMP_WORK
    "DAYS_EMPLOYED": "EMP_WORK",          # proxy (binary)
    "AMT_INCOME_TOTAL": "FINC_TOT_ND",
    "AMT_CREDIT":    "WLTH_ODEB_TOT_ND",
    "AMT_ANNUITY":   "EXPN_HOUS_TOT_ND",
    "AMT_GOODS_PRICE": "WLTH_VEHI_NET_ND",
    "REGION_POPULATION_RELATIVE": "GEO_METRO",
    "CNT_FAM_MEMBERS": "FAM_SIZE",
    "CNT_CHILDREN": "FAM_SIZE_CHI",
    "FLAG_OWN_REALTY": "HOME_STAT",
    "OCCUPATION_TYPE": "OCC_2010C_1M",
    # 필요 시 추가…
}

# 3) 특징 행렬 X 만들기
df_feat = df[[v for v in mapping.values() if v in df.columns]].copy()

# 전처리 예시 — 범주형/수치형 분리
cat_cols = df_feat.select_dtypes("object").columns.tolist()
num_cols = [c for c in df_feat.columns if c not in cat_cols]

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_cat = ohe.fit_transform(df_feat[cat_cols].fillna("MISSING"))
X_num = df_feat[num_cols].astype(np.float32).fillna(0).values
X = np.hstack([X_num, X_cat])

# 4) Target 생성   ― 예시: HomeCredit 모형 예측 기반
hc_model = joblib.load(hc_model_path)     # ← HomeCredit LightGBM
p_hat = hc_model.predict_proba(X)[:, 1]
tau = 0.08                                # PD 8% cut-off (조정 가능)
df['Default_Flag'] = (p_hat >= tau).astype(int)
y = df['Default_Flag'].values

# 5) PSID용 LightGBM 모델 학습 (5-fold CV)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros_like(y, dtype=np.float32)

params = dict(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    colsample_bytree=0.8,
    subsample=0.8,
    reg_lambda=0.1,
    random_state=42
)

for fold, (tr, val) in enumerate(kf.split(X), 1):
    clf = lgb.LGBMClassifier(**params)
    clf.fit(
        X[tr], y[tr],
        eval_set=[(X[val], y[val])],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(200, verbose=False),
                   lgb.log_evaluation(200)]
    )
    oof[val] = clf.predict_proba(X[val], num_iteration=clf.best_iteration_)[:, 1]
    gc.collect()

print("\n▶ 5-Fold CV AUC:", roc_auc_score(y, oof).round(5))

# 6) 확률 교정 (Isotonic 예)
calib = CalibratedClassifierCV(clf, cv="prefit", method="isotonic")
calib.fit(X, y)

# 7) 저장
joblib.dump(calib, DATA_DIR / "psid_credit_lgbm_calibrated.pkl")


AttributeError: 'OneHotEncoder' object has no attribute 'predict_proba'

In [7]:
"""
──────────────────────────────────────────────────────────────
 1) Home-Credit 모델(A) 로드   (OHE + 5-Fold LGBM 앙상블)
 2) PSID-SHELF 데이터 로드·가공 → 모델 입력 형태 맞추기
 3) 모델 예측 → psid_df['Default_Prob'] 생성
──────────────────────────────────────────────────────────────
author : you
date   : 2025-07-03
"""

# ────────────────────────────────────
# 라이브러리
# ────────────────────────────────────
import re, gc, joblib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

# ────────────────────────────────────
# 0. 경로 & 파일
# ────────────────────────────────────
DATA_DIR   = Path(r"D:/fintech/data")
MODEL_DIR  = Path(r"D:/fintech/models")
PSID_FILE  = DATA_DIR / "PSIDSHELF_1968_2021_WIDE.dta"
HC_TRAIN   = DATA_DIR / "home-credit-default-risk/application_train.csv"

OHE_FILE   = MODEL_DIR / "ohe_homecredit.pkl"
LGBM_PKLS  = sorted(MODEL_DIR.glob("lgbm_fold??.pkl"))   # 5개 꿰어오기

# ────────────────────────────────────
# 1. Home-Credit 측 정보 재현
# ────────────────────────────────────
print("┏▶ Home-Credit train 컬럼 재현 중 ...")
hc_df = pd.read_csv(HC_TRAIN, nrows=1)           # 헤더 용도로 1줄만
cat_cols = hc_df.select_dtypes("object").columns.tolist()
num_cols = hc_df.drop(columns=cat_cols + ["TARGET"]).columns.tolist()

ohe      = joblib.load(OHE_FILE)
feature_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
print(f"   ▷ 수치 {len(num_cols):>4d} / 범주 {len(cat_cols):>3d} "
      f"→ 전체 {len(feature_names):>4d}개")

# ────────────────────────────────────
# 2. PSID ↔ Home-Credit 컬럼 매핑 dict
# ────────────────────────────────────
MAP_HC2PSID = {
    # HomeCredit : PSID-SHELF (가장 최신 wave 기준)
    "EXT_SOURCE_1"          : "WLTH_TOT_DEB_ND_2021",
    "EXT_SOURCE_2"          : "WLTH_TOT_NET_ND_2021",
    "EXT_SOURCE_3"          : "FINC_TOT_ND_2021",
    "DAYS_BIRTH"            : "DEMO_AGE_GEN_2021",
    "DAYS_EMPLOYED"         : "EMP_WORK_2021",
    "AMT_CREDIT"            : "WLTH_ODEB_TOT_ND_2021",
    "AMT_ANNUITY"           : "EXPN_HOUS_TOT_ND_2021",
    "AMT_GOODS_PRICE"       : "WLTH_VEHI_NET_ND_2021",
    "AMT_INCOME_TOTAL"      : "FINC_TOT_ND_2021",
    "REGION_POPULATION_RELATIVE": "GEO_METRO_2021",
    "CNT_CHILDREN"          : "FAM_SIZE_CHI_2021",
    "CNT_FAM_MEMBERS"       : "FAM_SIZE_2021",
    "FLAG_OWN_REALTY"       : "HOME_STAT_2021",
    "CODE_GENDER"           : "DEMO_SEX",
    "NAME_EDUCATION_TYPE"   : "EDU_LEVEL_2021",
    "NAME_INCOME_TYPE"      : "EMP_STAT_1M_2021",
    "OCCUPATION_TYPE"       : "OCC_2010C_1M_2021",
    # ↓ HomeCredit엔 있지만 PSID에서 proxy 계산이 필요한 컬럼
    "FLAG_OWN_CAR"          : "WLTH_VEHI_NET_ND_2021"   # >0 → 1
}

# ────────────────────────────────────
# 3. PSID 데이터 가공 함수
# ────────────────────────────────────
def prepare_psid(psid_raw: pd.DataFrame) -> pd.DataFrame:
    """PSID 원본 → Home-Credit 스키마(DataFrame) 반환"""
    df = pd.DataFrame(index=psid_raw.index)

    # 3-1) 단순 매핑
    for hc_col, psid_col in MAP_HC2PSID.items():
        if psid_col in psid_raw.columns:
            df[hc_col] = psid_raw[psid_col]
        else:                                  # 없는 연도면 가장 최근 연도 찾아 채움
            base = re.sub(r"_\d{4}$", "", psid_col)
            cand = sorted([c for c in psid_raw.columns if c.startswith(base)],
                          reverse=True)
            df[hc_col] = psid_raw[cand].bfill(axis=1).iloc[:, 0]

    # 3-2) 파생 (차량 보유 더미)
    df["FLAG_OWN_CAR"] = (df["FLAG_OWN_CAR"] > 0).astype(int)

    # 3-3) 범주형‧수치 분리
    df_cat = df[cat_cols].astype("category")
    df_num = df.drop(columns=cat_cols).astype(np.float32)

    # 3-4) OHE 변환 & 스택
    X_num  = df_num.values
    X_cat  = ohe.transform(df_cat).astype(np.float32)
    X_mat  = np.hstack([X_num, X_cat])
    return df, X_mat

# ────────────────────────────────────
# 4. 모델 불러와 예측
# ────────────────────────────────────
print("┏▶ PSID 데이터 로드 중 ...")
psid_raw = pd.read_stata(PSID_FILE, convert_categoricals=False)

# 필요한 열만 살짝 추려서 메모리↓
need_cols = set(MAP_HC2PSID.values())
for base in ["WLTH_VEHI_NET_ND", "WLTH_TOT_DEB_ND", "WLTH_TOT_NET_ND",
             "FINC_TOT_ND", "EMP_STAT_1M", "OCC_2010C_1M",
             "EXPN_HOUS_TOT_ND", "FAM_SIZE", "FAM_SIZE_CHI",
             "HOME_STAT", "DEMO_AGE_GEN", "EDU_LEVEL", "GEO_METRO"]:
    need_cols.update([c for c in psid_raw.columns if c.startswith(base)])

psid_raw = psid_raw[list(need_cols | {"DEMO_SEX"})].copy()

print(f"   ▷ shape = {psid_raw.shape}")

# 4-1) 전처리
psid_df, X = prepare_psid(psid_raw)

# 4-2) 5-Fold 모델 예측 (평균)
print("┏▶ 5-Fold LGBM 예측 중 ...")
preds = np.zeros(X.shape[0], dtype=np.float32)

for pkl in LGBM_PKLS:
    clf: LGBMClassifier = joblib.load(pkl)
    preds += clf.predict_proba(X, num_iteration=clf.best_iteration_)[:, 1] \
             / len(LGBM_PKLS)
    gc.collect()
    
# 4-3) 결측이 너무 많은 행 필터링 & CSV 저장
# 'MAP_HC2PSID'에 매핑된 PSID 컬럼들이 모두 NaN인 행은 제외
hc_inputs = list(MAP_HC2PSID.keys())
mask_any = psid_df[hc_inputs].notna().any(axis=1)

psid_valid = psid_df[mask_any].copy()
print(f"전체 {len(psid_df)}명 중, 유효 데이터 {len(psid_valid)}명 → CSV 저장")

OUT_CSV = DATA_DIR / "psid_valid_for_model.csv"
psid_valid.to_csv(OUT_CSV, index=False)
print(f"✅ 저장 완료 → {OUT_CSV}")


psid_df["Default_Prob"] = preds

print("   ▷ 완료! 예시 출력:")
print(psid_df[["Default_Prob"]].head())

# ────────────────────────────────────
# 5. 저장
# ────────────────────────────────────
OUT_FILE = DATA_DIR / "psid_with_default_prob.parquet"
psid_df.to_parquet(OUT_FILE, index=False)
print(f"\n✅  결과 저장 → {OUT_FILE}")


┏▶ Home-Credit train 컬럼 재현 중 ...
   ▷ 수치  105 / 범주  16 → 전체  251개
┏▶ PSID 데이터 로드 중 ...


KeyError: "['WLTH_ODEB_TOT_ND_2021'] not in index"

In [14]:
import pandas as pd

# 1) HomeCredit ↔ PSID 매핑
MAP_HC2PSID = {
    "EXT_SOURCE_1":          "WLTH_TOT_DEB_ND_2021",
    "EXT_SOURCE_2":          "WLTH_TOT_NET_ND_2021",
    "EXT_SOURCE_3":          "FINC_TOT_ND_2021",
    "DAYS_BIRTH":            "DEMO_AGE_GEN_2021",
    "DAYS_EMPLOYED":         "EMP_WORK_2021",
    "AMT_CREDIT":            "WLTH_ODEB_TOT_ND_2021",
    "AMT_ANNUITY":           "EXPN_HOUS_TOT_ND_2021",
    "AMT_GOODS_PRICE":       "WLTH_VEHI_NET_ND_2021",
    "AMT_INCOME_TOTAL":      "FINC_TOT_ND_2021",
    "REGION_POPULATION_RELATIVE": "GEO_METRO_2021",
    "CNT_CHILDREN":          "FAM_SIZE_CHI_2021",
    "CNT_FAM_MEMBERS":       "FAM_SIZE_2021",
    "FLAG_OWN_REALTY":       "HOME_STAT_2021",
    "CODE_GENDER":           "DEMO_SEX",
    "NAME_EDUCATION_TYPE":   "EDU_LEVEL_2021",
    "NAME_INCOME_TYPE":      "EMP_STAT_1M_2021",
    "OCCUPATION_TYPE":       "OCC_2010C_1M_2021",
    # proxy 계산용
    "FLAG_OWN_CAR":          "WLTH_VEHI_NET_ND_2021",
}

# 2) 읽어올 PSID 칼럼 목록 생성 (ID 포함)
psid_cols = set(MAP_HC2PSID.values())  # 실제 있는 PSID 칼럼들
psid_cols.add("ID")                    # 식별용 ID
use_cols = list(psid_cols)

# 3) 필요한 칼럼만 읽기
psid = pd.read_stata(
    "D:/fintech/data/PSIDSHELF_1968_2021_WIDE.dta",
    columns=use_cols,
    convert_categoricals=False
)

# 4) 빈 DataFrame 에 매핑
df = pd.DataFrame({"ID": psid["ID"]})
for hc_col, psid_col in MAP_HC2PSID.items():
    if psid_col not in psid.columns:
        print(f"⚠️ PSID에 {psid_col} 컬럼이 없습니다.")
        df[hc_col] = pd.NA
    else:
        df[hc_col] = psid[psid_col]

# 5) FLAG_OWN_CAR proxy 계산 (>0 → 1)
if "WLTH_VEHI_NET_ND_2021" in psid.columns:
    df["FLAG_OWN_CAR"] = (psid["WLTH_VEHI_NET_ND_2021"] > 0).astype(int)
else:
    print("⚠️ WLTH_VEHI_NET_ND_2021 이 없어 FLAG_OWN_CAR 계산 불가.")

# 6) 하나라도 NaN 이면 제거 (모두 채워진 행만 남김)
keep = df.drop(columns="ID").notna().all(axis=1)
df_valid = df.loc[keep].reset_index(drop=True)

# 7) CSV 로 저장
out_path = "psid_for_homecredit_input.csv"
df_valid.to_csv(out_path, index=False)
print(f"완료: {out_path} (shape={df_valid.shape})")


MemoryError: 

In [21]:
# df: Stata에서 한 번에 다 읽어들일 수 없어서 메타만 읽었던 df_meta 가 아니라
#      실제 데이터를 chunk 로 나눠서 읽어야 메모리 에러를 피할 수 있습니다.

import pandas as pd

# 예: Stata 파일을 iterator 로 조금씩 읽어서 컬럼만 한 번 뽑아보고
reader = pd.read_stata(
    "D:/fintech/data/PSIDSHELF_1968_2021_WIDE.dta",
    convert_categoricals=False,
    iterator=True,
    chunksize=1  # 헤더만 볼 수 있다면 충분합니다
)
df_chunk = next(reader)  # 컬럼 정보만 필요
print("전체 컬럼 수:", len(df_chunk.columns))
print("예시로 WLTH_VEHI_NET_ND_* 컬럼들:", 
      [c for c in df_chunk.columns if c.startswith("WLTH_VEHI_NET_ND_")])


전체 컬럼 수: 7525
예시로 WLTH_VEHI_NET_ND_* 컬럼들: ['WLTH_VEHI_NET_ND_1984', 'WLTH_VEHI_NET_ND_1989', 'WLTH_VEHI_NET_ND_1994', 'WLTH_VEHI_NET_ND_1999', 'WLTH_VEHI_NET_ND_2001', 'WLTH_VEHI_NET_ND_2003', 'WLTH_VEHI_NET_ND_2005', 'WLTH_VEHI_NET_ND_2007', 'WLTH_VEHI_NET_ND_2009', 'WLTH_VEHI_NET_ND_2011', 'WLTH_VEHI_NET_ND_2013', 'WLTH_VEHI_NET_ND_2015', 'WLTH_VEHI_NET_ND_2017', 'WLTH_VEHI_NET_ND_2019', 'WLTH_VEHI_NET_ND_2021']


In [29]:
import pandas as pd
from pandas.io.stata import StataReader

FILE = "D:/fintech/data/PSIDSHELF_1968_2021_WIDE.dta"

# 1) StataReader 로 변수 목록만 확보
reader = StataReader(FILE)
all_cols = set(reader.varlist)

# 2) 관심 있는 접두사·연도 조합 생성
prefixes = ["EARN_TOT_ND", "FINC_TOT_ND", "WLTH_TOT_NET_ND",
            "WLTH_TOT_DEB_ND", "WLTH_VEHI_NET_ND", "EXPN_HOUS_TOT_ND"]
years = range(1990, 2022)
wanted = {f"{p}_{y}" for p in prefixes for y in years}

# 실제로 존재하는 칼럼만 필터
available = sorted(all_cols & wanted)
always = ["ID", "PNUM", "LINEAGE"]
columns_to_read = always + available

print(f"읽을 컬럼 ({len(columns_to_read)}개):\n", columns_to_read)

# 3) 전체 읽어서(불가피) 필요한 칼럼만 슬라이스
df_full = reader.read()
df = df_full[columns_to_read]

print("결과 shape:", df.shape)
print(df.head())


AttributeError: 'StataReader' object has no attribute 'varlist'

In [34]:
import pandas as pd
from pandas.io.stata import StataReader

# 1) 파일 경로, 관심 접두사·연도 정의
FILE = "D:/fintech/data/PSIDSHELF_1968_2021_WIDE.dta"
prefixes = [
    "EARN_TOT_ND", "FINC_TOT_ND",
    "WLTH_TOT_NET_ND", "WLTH_TOT_DEB_ND",
    "WLTH_VEHI_NET_ND", "EXPN_HOUS_TOT_ND"
]
years = list(range(1968, 2022))

# 2) StataReader로 메타만 0행 읽기 → 변수명 리스트 확보
reader = StataReader(FILE)
meta = reader.read(nrows=0)        # zero rows read → 메타데이터만
all_cols = list(meta.columns)      # 모든 변수명 리스트

# 3) 접두사+연도 조합 중 실제 존재하는 변수만 골라내기
target_cols = [
    f"{pref}_{yr}"
    for pref in prefixes
    for yr in years
    if f"{pref}_{yr}" in all_cols
]

# 4) 식별자 id 변수들(예: ID, PNUM, LINEAGE 등) + target_cols 만 읽기
id_cols = ["ID", "PNUM", "LINEAGE"]
use_cols = id_cols + target_cols

# 5) 실제 데이터 로드 (columns 파라미터로 필요한 변수만)
df = pd.read_stata(FILE, columns=use_cols)

# 결과 확인
print(df.shape)
print(df.columns.tolist())


StopIteration: 

In [35]:
# 이미 StataReader로 meta.columns 를 뽑아둔 상태라고 가정
all_cols = meta.columns.tolist()

# 관심 접두사만 나열
prefixes = [
    "EARN_TOT_ND",
    "FINC_TOT_ND",
    "WLTH_TOT_NET_ND",
    "WLTH_TOT_DEB_ND",
    "WLTH_VEHI_NET_ND",
    "EXPN_HOUS_TOT_ND",
]

# ID 변수와, 접두사로 시작하는 변수만 필터링
id_vars = ["ID", "PNUM", "LINEAGE"]
target_cols = [
    col for col in all_cols
    if any(col.startswith(pref + "_") for pref in prefixes)
]

use_cols = id_vars + target_cols

print("선택된 변수들:")
for c in use_cols:
    print(" ", c)


NameError: name 'meta' is not defined

In [ ]:
earn_tot_nd_columns = [
    "EARN_TOT_ND_1968",
    "EARN_TOT_ND_1969",
    "EARN_TOT_ND_1970",
    "EARN_TOT_ND_1971",
    "EARN_TOT_ND_1972",
    "EARN_TOT_ND_1973",
    "EARN_TOT_ND_1974",
    "EARN_TOT_ND_1975",
    "EARN_TOT_ND_1976",
    "EARN_TOT_ND_1977",
    "EARN_TOT_ND_1978",
    "EARN_TOT_ND_1979",
    "EARN_TOT_ND_1980",
    "EARN_TOT_ND_1981",
    "EARN_TOT_ND_1982",
    "EARN_TOT_ND_1983",
    "EARN_TOT_ND_1984",
    "EARN_TOT_ND_1985",
    "EARN_TOT_ND_1986",
    "EARN_TOT_ND_1987",
    "EARN_TOT_ND_1988",
    "EARN_TOT_ND_1989",
    "EARN_TOT_ND_1990",
    "EARN_TOT_ND_1991",
    "EARN_TOT_ND_1992",
    "EARN_TOT_ND_1993",
    "EARN_TOT_ND_1994",
    "EARN_TOT_ND_1995",
    "EARN_TOT_ND_1996",
    "EARN_TOT_ND_1997",
    "EARN_TOT_ND_1999",
    "EARN_TOT_ND_2001",
    "EARN_TOT_ND_2003",
    "EARN_TOT_ND_2005",
    "EARN_TOT_ND_2007",
    "EARN_TOT_ND_2009",
    "EARN_TOT_ND_2011",
    "EARN_TOT_ND_2013",
    "EARN_TOT_ND_2015",
    "EARN_TOT_ND_2017",
    "EARN_TOT_ND_2019",
    "EARN_TOT_ND_2021",
]


In [3]:
import pandas as pd
from pathlib import Path
DATA_DIR = Path(r"D:/fintech/data")   
SRC_FILE = DATA_DIR / "PSIDSHELF_1968_2021_LONG.dta"
df = pd.read_stata(SRC_FILE, convert_categoricals=False)
df.loc[1:4,earn_tot_nd_columns]

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\IPython\core\interactiveshell.py", line 3550, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_14500\3120647646.py", line 5, in <module>
    df = pd.read_stata(SRC_FILE, convert_categoricals=False)
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\pandas\io\stata.py", line 2109, in read_stata
    return reader.read()
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\pandas\io\stata.py", line 1785, in read
    data = self._do_convert_missing(data, convert_missing)
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\pandas\io\stata.py", line 1866, in _do_convert_missing
    data.isetitem(idx, value)
  File "C:\Users\Admin\miniforge3\envs\fintech\lib\site-packages\pandas\core\frame.py", line 4268, in isetitem
    arraylike, refs = self._sanitize_column(value)
  File "C:\Users\Admi

In [ ]:
import pandas as pd

df = pd.read_csv('D:/fintech/data/PSIDSHELF_1968_2021_LONG.csv')
df.head()

In [5]:
import pandas as pd

df2 = pd.read_csv('./mnt/d/fintech/data/PSIDSHELF_1968_2021_LONG.csv')
df2.head()

FileNotFoundError: [Errno 2] No such file or directory: './mnt/d/fintech/data/PSIDSHELF_1968_2021_LONG.csv'